# MuraltZ Sandbox Segmentation — YOLOv8l-seg Training

**Target:** ~3–3.5 hours on T4 GPU  
**Model:** YOLOv8l-seg (46M params)  
**Unified classes:** crater, rock, shadow, boulder  
**Dataset:** 83 annotated sandbox images (merged from Dataset A + B)  
**Output:** `sandbox_seg.pt` → download to `ground_station/models/`

### Before you start
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `yolo_training_datasets.zip` (5.3 MB) when prompted
3. Run cells top to bottom — training takes ~3 hours

In [ ]:
# Cell 1 — Install & verify GPU
!pip install -q ultralytics
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Runtime → Change runtime type → T4 GPU")

In [ ]:
# Cell 2 — Mount Google Drive (checkpoints saved here so you never lose them)
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/MuraltZ_Models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Backup dir: {SAVE_DIR}")

In [ ]:
# Cell 3 — Upload dataset zip
from google.colab import files
print("Upload yolo_training_datasets.zip (5.3 MB)")
uploaded = files.upload()

In [ ]:
# Cell 4 — Unzip and merge datasets with class remapping
import os, shutil, yaml

!unzip -q -o yolo_training_datasets.zip -d /content/raw

# Dataset A: 0=crater, 1=rock, 2=shadow → keep as-is
# Dataset B: 0=Crater, 1=Rock, 2=boulder → 0=crater, 1=rock, 2→3=boulder
A_MAP = {0: 0, 1: 1, 2: 2}
B_MAP = {0: 0, 1: 1, 2: 3}
CLASSES = ['crater', 'rock', 'shadow', 'boulder']

MERGED = '/content/dataset'
if os.path.exists(MERGED):
    shutil.rmtree(MERGED)

total = 0
for split in ['train', 'valid', 'test']:
    for ds, cmap in [('dataset_A', A_MAP), ('dataset_B', B_MAP)]:
        src_img = f'/content/raw/{ds}/{split}/images'
        src_lbl = f'/content/raw/{ds}/{split}/labels'
        dst_img = f'{MERGED}/{split}/images'
        dst_lbl = f'{MERGED}/{split}/labels'
        os.makedirs(dst_img, exist_ok=True)
        os.makedirs(dst_lbl, exist_ok=True)
        if not os.path.exists(src_img):
            continue
        # Copy images
        for f in os.listdir(src_img):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                shutil.copy2(os.path.join(src_img, f), os.path.join(dst_img, f))
                total += 1
        # Remap labels
        if os.path.exists(src_lbl):
            for f in os.listdir(src_lbl):
                if not f.endswith('.txt'):
                    continue
                with open(os.path.join(src_lbl, f)) as fh:
                    lines = []
                    for line in fh:
                        parts = line.strip().split()
                        if not parts:
                            continue
                        old_cls = int(parts[0])
                        new_cls = cmap.get(old_cls, old_cls)
                        lines.append(f"{new_cls} {' '.join(parts[1:])}")
                with open(os.path.join(dst_lbl, f), 'w') as fh:
                    fh.write('\n'.join(lines) + '\n')

# Write data.yaml
data_cfg = {
    'train': f'{MERGED}/train/images',
    'val': f'{MERGED}/valid/images',
    'nc': len(CLASSES),
    'names': CLASSES,
}
yaml_path = f'{MERGED}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_cfg, f)

# Report
for split in ['train', 'valid', 'test']:
    d = f'{MERGED}/{split}/images'
    n = len(os.listdir(d)) if os.path.exists(d) else 0
    print(f"  {split}: {n} images")
print(f"\nTotal: {total} images")
print(f"Classes: {CLASSES}")
print(f"Config: {yaml_path}")

In [ ]:
# Cell 5 — Train YOLOv8l-seg (~3 hours on T4)
#
# l-seg vs x-seg: 46M vs 72M params, ~40% faster training
# For 4 classes on 83 images the accuracy gap is negligible
# 200 epochs with patience=40 — your x-seg peaked around epoch 350-400
# l-seg converges faster, so 200 is plenty

from ultralytics import YOLO

model = YOLO('yolov8l-seg.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=200,
    imgsz=1024,
    batch=4,
    device=0,
    workers=2,
    patience=40,
    save=True,
    save_period=25,
    project='/content/runs',
    name='sandbox_seg',
    exist_ok=True,
    # Heavy augmentation (critical for small dataset)
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=15.0,
    translate=0.1,
    scale=0.4,
    fliplr=0.5,
    flipud=0.2,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    erasing=0.3,
    # Optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,
)

print('\n' + '=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)

In [ ]:
# Cell 6 — Backup to Google Drive + download
import shutil
from google.colab import files

best_pt = '/content/runs/sandbox_seg/weights/best.pt'
last_pt = '/content/runs/sandbox_seg/weights/last.pt'

if os.path.exists(best_pt):
    shutil.copy2(best_pt, os.path.join(SAVE_DIR, 'sandbox_seg.pt'))
    shutil.copy2(last_pt, os.path.join(SAVE_DIR, 'sandbox_seg_last.pt'))
    sz = os.path.getsize(best_pt) / 1024 / 1024
    print(f"Saved to Google Drive: {SAVE_DIR}/sandbox_seg.pt ({sz:.1f} MB)")
    print("\nDownloading best.pt...")
    files.download(best_pt)
else:
    print("ERROR: best.pt not found!")
    !find /content/runs -name '*.pt' 2>/dev/null

In [ ]:
# Cell 7 — Quick validation (optional)
from ultralytics import YOLO
model = YOLO('/content/runs/sandbox_seg/weights/best.pt')
metrics = model.val(data='/content/dataset/data.yaml', imgsz=1024, device=0)
print(f"\nBox  mAP50:    {metrics.box.map50:.3f}")
print(f"Box  mAP50-95: {metrics.box.map:.3f}")
print(f"Mask mAP50:    {metrics.seg.map50:.3f}")
print(f"Mask mAP50-95: {metrics.seg.map:.3f}")
if metrics.seg.map50 > 0.85:
    print("\nModel is GOOD — ready for deployment!")
elif metrics.seg.map50 > 0.70:
    print("\nModel is DECENT — usable.")
else:
    print("\nModel needs more data or training.")